# Real-world False Positive Rate

Capture normal traffic on the firewall VM, then run this notebook top-to-bottom.

```bash
sudo tcpdump -i <iface> -w real_traffic.pcap
```

Browse normal sites such as YouTube, GitHub, Wikipedia, and Gmail for about 30 minutes, then press Ctrl+C. Copy the capture to `data/real_traffic.pcap` in this repo.

In [ ]:
from pathlib import Path
from collections import Counter
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scapy.all import IP, TCP, UDP, rdpcap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models/metrics.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PCAP_PATH = PROJECT_ROOT / "data/real_traffic.pcap"
FIGURES_DIR = PROJECT_ROOT / "docs/report/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
if not PCAP_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PCAP_PATH}. Capture normal traffic with tcpdump and copy it to data/real_traffic.pcap."
    )

pkts = rdpcap(str(PCAP_PATH))
print(f"Loaded {len(pkts):,} packets from {PCAP_PATH}")
if len(pkts) > 500_000:
    random.seed(42)
    pkts = random.sample(list(pkts), 500_000)
    pkts.sort(key=lambda pkt: float(getattr(pkt, "time", 0.0)))
    print("Sampled down to 500,000 packets for notebook runtime.")

In [ ]:
from ngfw.config import load_config
from ngfw.flow_builder import FlowBuilder

cfg = load_config()
closed_flows = []
builder = FlowBuilder(cfg, closed_flows.append)
last_ts = 0.0

def parse_packet(pkt):
    if IP not in pkt:
        return None
    ip = pkt[IP]
    proto = "TCP" if TCP in pkt else ("UDP" if UDP in pkt else "OTHER")
    src_port = pkt[TCP].sport if TCP in pkt else (pkt[UDP].sport if UDP in pkt else 0)
    dst_port = pkt[TCP].dport if TCP in pkt else (pkt[UDP].dport if UDP in pkt else 0)
    tcp_flags = int(pkt[TCP].flags) if TCP in pkt else 0
    iface = getattr(pkt, "sniffed_on", None) or cfg.interfaces[0]
    return {
        "ts": float(pkt.time),
        "src_ip": ip.src,
        "dst_ip": ip.dst,
        "src_port": src_port,
        "dst_port": dst_port,
        "proto": proto,
        "length": int(ip.len) if ip.len is not None else len(ip),
        "tcp_flags": tcp_flags,
        "interface": iface,
    }

for pkt in pkts:
    parsed = parse_packet(pkt)
    if parsed is None:
        continue
    last_ts = max(last_ts, parsed["ts"])
    builder.add_packet(parsed)

builder.sweep_timeouts(now=last_ts + 200)
print(f"Closed flows: {len(closed_flows):,}")

In [ ]:
from ngfw.feature_extractor import extract

features = [extract(flow) for flow in closed_flows]
if not features:
    raise RuntimeError("No IP flows reconstructed from the pcap.")
X = np.vstack(features)
print(f"Feature matrix: {X.shape}")

In [ ]:
from ngfw.classifier import Classifier

clf = Classifier(load_config())
predictions = []
for vec in X:
    label, conf = clf.predict(vec)
    predictions.append((label, conf))

print(f"Scored flows: {len(predictions):,}")

In [ ]:
THRESHOLD = 0.85
total = len(predictions)
false_positives = [(label, conf) for label, conf in predictions if label != "BENIGN" and conf >= THRESHOLD]
fpr = len(false_positives) / total if total else 0.0
pred_counts = Counter(label for label, _conf in predictions)
fp_counts = Counter(label for label, _conf in false_positives)

print(f"Total flows = {total:,}")
print(f"False positives at confidence >= {THRESHOLD:.2f} = {len(false_positives):,}")
print(f"FPR = {fpr:.4%}")
print(dict(pred_counts))

## Domain-shift discussion

If the measured FPR is above 5%, report it honestly. A likely cause is domain shift: CICIDS2017 BENIGN traffic is a lab-era mix, while modern browsing is dominated by encrypted HTTPS, CDN-heavy flows, browser multiplexing, and background application traffic. This notebook does not tune the threshold to hide false positives; it surfaces the measured result for the report.

In [ ]:
plot_labels = [label for label in clf.labels if label in pred_counts]
plot_values = [pred_counts[label] for label in plot_labels]

plt.figure(figsize=(7, 4))
sns.barplot(x=plot_labels, y=plot_values, color="#4C78A8")
plt.title(f"Real-world traffic predictions, FPR={fpr:.2%}")
plt.xlabel("Predicted class")
plt.ylabel("Flow count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fpr_summary.png", dpi=150, bbox_inches="tight")
plt.close()
print(FIGURES_DIR / "fpr_summary.png")

In [ ]:
print(f"FPR = {fpr:.2%}")